# SVD-Based Block Encoding of the Chebyshev Differentiation Matrix

## Theory, Construction, and Circuit Optimisation

This notebook builds a **block-encoding quantum circuit** for the shifted Chebyshev
spectral-differentiation matrix $A = D^{(\text{cheb})} - \alpha I$, using an SVD
decomposition that yields circuits roughly **half the depth** of a naïve dilation.

### Outline

1. **Background** — block encodings and why they matter
2. **The Chebyshev differentiation matrix** — definition and structure
3. **SVD decomposition** — factoring the block encoding into three layers
4. **Uniformly Controlled $R_y$ (UCRy)** — Gray-code decomposition
5. **Depth analysis** — why the 4×4 case gives depth 17
6. **Generalised implementation** — parameterised by $n$ and $\alpha$
7. **Comparison** — SVD vs direct dilation
8. **Normalisation factor** — recovering $A$ from the circuit

## 1. Block Encodings

A matrix $A \in \mathbb{C}^{n \times n}$ is in general **not unitary**, so it cannot be
directly implemented as a quantum gate. A *block encoding* embeds $A/\alpha$ into the
top-left corner of a larger unitary $U_{BE}$ acting on $n_{\text{sys}} + n_{\text{anc}}$
qubits:

$$\langle 0_a | \, U_{BE} \, | 0_a \rangle = \frac{A}{\alpha}$$

where $|0_a\rangle$ denotes the ancilla register in the all-zero state and
$\alpha \ge \|A\|_2$ is the **sub-normalisation factor** (the spectral norm,
i.e. the largest singular value).

Block encodings are the standard interface for **Quantum Singular Value
Transformation (QSVT)**, Hamiltonian simulation, and quantum linear-systems
algorithms. The quality of the block encoding — its depth, gate count, and
ancilla overhead — directly determines the practical cost of these algorithms.

## 2. The Chebyshev Spectral Differentiation Matrix

Given a polynomial of degree $n$ expanded in the Chebyshev basis
$p(x) = \sum_{k=0}^{n} c_k T_k(x)$, its derivative has coefficients related by:

$$c_k^\prime = \frac{2}{\sigma_k} \sum_{\substack{j = k+1 \\ j+k \text{ odd}}}^{n} j \, c_j, \qquad \sigma_k = \begin{cases} 2 & k = 0 \\ 1 & k \ge 1 \end{cases}$$

This defines the $(n{+}1) \times (n{+}1)$ matrix $D$ such that $\mathbf{c}^\prime = D\,\mathbf{c}$.

### Key properties

| Property | Implication |
|----------|------------|
| **Strictly upper triangular** | $D$ is nilpotent: $D^{n+1} = 0$ |
| **Structured sparsity** | Non-zero only when $j > k$ and $j + k$ is odd |
| **Shift** $A = D - \alpha I$ | Models $\partial_x - \alpha$ (advection/decay) |
| **Dimension** $m = n+1$ | Padded to $m_{\text{pad}} = 2^q$ for the quantum circuit |

For the quantum circuit we need $m_{\text{pad}} = 2^{\lceil \log_2 m \rceil}$ to be a
power of two. Padded rows/columns carry $-\alpha$ on the diagonal (consistent with the shift).

In [16]:
import numpy as np
from scipy.linalg import svd, norm
from qiskit import QuantumCircuit
from qiskit.quantum_info import Operator
from qiskit.circuit.library import UnitaryGate
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from itertools import permutations
import math

def D_cheb_coeff(n):
    j = np.arange(n+1)[None, :]
    k = np.arange(n+1)[:, None]
    sigma = np.ones((n+1, 1)); sigma[0, 0] = 2.0
    mask = (j > k) & (((j + k) % 2) == 1)
    D = np.zeros((n+1, n+1))
    D[mask] = (2.0 * j / sigma)[mask]
    return D

# Display for n = 3 (the 4x4 case)
D3 = D_cheb_coeff(3)
print("D_cheb(3) -- Chebyshev differentiation matrix:")
print(D3)
print(f"\nStrictly upper triangular: {np.allclose(np.tril(D3), 0)}")
print(f"Nilpotent (D^4 = 0):      {np.allclose(np.linalg.matrix_power(D3, 4), 0)}")

D_cheb(3) -- Chebyshev differentiation matrix:
[[0. 1. 0. 3.]
 [0. 0. 4. 0.]
 [0. 0. 0. 6.]
 [0. 0. 0. 0.]]

Strictly upper triangular: True
Nilpotent (D^4 = 0):      True


In [17]:
def build_matrix(n, alpha):
    m = n + 1
    q = int(np.ceil(np.log2(m)))
    m_pad = 2**q
    
    D = D_cheb_coeff(n)
    A = D - alpha * np.eye(m)
    
    if m < m_pad:
        A_pad = np.zeros((m_pad, m_pad))
        A_pad[:m, :m] = A
        for i in range(m, m_pad):
            A_pad[i, i] = -alpha
    else:
        A_pad = A
    
    return A_pad, m, m_pad

# Example: n=3, alpha=5
n, alpha = 3, 5.0
A, m, m_pad = build_matrix(n, alpha)

print(f"n = {n},  alpha = {alpha}")
print(f"Original size: {m}x{m},  Padded size: {m_pad}x{m_pad}")
print(f"\nA = D_cheb({n}) - {alpha}*I:")
print(A)

svals = svd(A, compute_uv=False)
print(f"\nSingular values: {np.round(svals, 6)}")
print(f"||A||_2 = {svals[0]:.6f}")

n = 3,  alpha = 5.0
Original size: 4x4,  Padded size: 4x4

A = D_cheb(3) - 5.0*I:
[[-5.  1.  0.  3.]
 [ 0. -5.  4.  0.]
 [ 0.  0. -5.  6.]
 [ 0.  0.  0. -5.]]

Singular values: [9.701731 6.24103  4.959191 2.081439]
||A||_2 = 9.701731


## 3. SVD Block-Encoding Decomposition

The standard way to build a 1-ancilla block encoding is the **direct dilation**:

$$U_{BE}^{\text{direct}} = \begin{pmatrix} A/\alpha & \sqrt{I - (A/\alpha)(A/\alpha)^\dagger} \\ \sqrt{I - (A/\alpha)^\dagger(A/\alpha)} & -(A/\alpha)^\dagger \end{pmatrix}$$

This $2n \times 2n$ unitary is treated as a **black box** by the transpiler, which
must use generic decomposition (QSD) requiring up to $O(4^k)$ CNOT gates for $k$ qubits.

### The SVD approach

Instead, we exploit $A = U_A \Sigma V^\dagger$ to factor the block encoding into **three structured layers**:

$$U_{BE} = (I_a \otimes U_A) \cdot \text{UCRy}(\boldsymbol{\theta}) \cdot (I_a \otimes V^\dagger)$$

**Why this works:** Starting from $|j\rangle|0_a\rangle$:

1. $V^\dagger$ transforms the system to the SVD basis
2. UCRy rotates the ancilla by $\theta_x = 2\arccos(\sigma_x/\alpha)$ for system state $|x\rangle$
3. $U_A$ applies the left singular vectors

Projecting onto $\langle 0_a|$:

$$\langle 0_a | U_{BE} | 0_a \rangle = U_A \cdot \mathrm{diag}\big(\cos(\theta_x/2)\big) \cdot V^\dagger = U_A \cdot \mathrm{diag}\big(\sigma_x/\alpha\big) \cdot V^\dagger = \frac{A}{\alpha}$$

### CX gate count (4x4 case)

| Component | Operation | CX gates |
|-----------|-----------|----------|
| $V^\dagger$ | 2-qubit unitary | $\le 3$ (typically 2) |
| UCRy | $N$ controlled-$R_y$ | $N = 2^{n_{\text{sys}}}$ (= 4) |
| $U_A$ | 2-qubit unitary | $\le 3$ (typically 2) |
| **Total** | | **8** |

Compare with **direct dilation**: opaque 3-qubit unitary -> **18-20 CX** via QSD.

## 4. Uniformly Controlled $R_y$ -- Gray-Code Decomposition

The UCRy must apply a **different** $R_y(\theta_x)$ rotation to the ancilla for each
of the $N = 2^{n_{\text{sys}}}$ computational basis states $|x\rangle$ of the system.

### Gray-code decomposition: $N$ single-qubit $R_y$ + $N$ CX

The efficient decomposition interleaves $R_y$ gates with CX gates whose controls
follow the **Gray-code transition sequence**:

$$R_y(a_0) \to \text{CX}(c_0) \to R_y(a_1) \to \text{CX}(c_1) \to \cdots \to R_y(a_{N-1}) \to \text{CX}(\text{close})$$

For each state $|x\rangle$, the CX gates selectively flip the ancilla, creating a sign pattern.
The effective rotation angle is:

$$\theta_{\text{eff}}(x) = \sum_{k=0}^{N-1} a_k \cdot (-1)^{f_k(x)}$$

This defines a **sign matrix** $M$ with $M_{x,k} = (-1)^{f_k(x)}$, and the decomposed
angles are found by solving $M \cdot \mathbf{a} = \boldsymbol{\theta}$.

### For $n_{\text{sys}} = 2$ (4x4 matrix)

Gray code: $0 \to 1 \to 3 \to 2$. Transition bits: $0, 1, 0$. Closing bit: $1$.

CX sequence: CX($q_0$, anc), CX($q_1$, anc), CX($q_0$, anc), CX($q_1$, anc).

This gives **4 CX + 4 $R_y$** = fixed depth 7 for the UCRy layer.

In [ ]:
# =====================================================================
# CORE BLOCK-ENCODING FUNCTIONS
# =====================================================================

def _gray_code_control_sequence(n_sys_qubits):
    """Gray-code CX control sequence for UCRy decomposition."""
    N = 2 ** n_sys_qubits
    gray = lambda x: x ^ (x >> 1)
    
    ctrl_seq = []
    for k in range(1, N):
        changed = gray(k) ^ gray(k - 1)
        ctrl_seq.append(int(np.log2(changed)))
    
    gray_last = gray(N - 1)
    closing_bit = int(np.log2(gray_last)) if gray_last > 0 else 0
    return ctrl_seq, closing_bit


def build_ucry_angles(thetas, n_sys_qubits):
    """Compute decomposed Gray-code UCRy rotation angles by solving M*a = theta."""
    N = 2 ** n_sys_qubits
    ctrl_seq, _ = _gray_code_control_sequence(n_sys_qubits)
    
    M = np.zeros((N, N))
    for x in range(N):
        ancilla = 0
        for k in range(N):
            M[x, k] = (-1) ** ancilla
            if k < len(ctrl_seq):
                if (x >> ctrl_seq[k]) & 1:
                    ancilla ^= 1
    
    return np.linalg.solve(M, thetas)


def build_ucry_circuit(qc, a_ucry, sys_qubits, anc_qubit, n_sys_qubits):
    """Append the UCRy sub-circuit using Gray-code CX ordering."""
    N = 2 ** n_sys_qubits
    ctrl_seq, closing_bit = _gray_code_control_sequence(n_sys_qubits)
    
    qc.ry(float(a_ucry[0]), anc_qubit)
    for k in range(N - 1):
        qc.cx(sys_qubits[ctrl_seq[k]], anc_qubit)
        qc.ry(float(a_ucry[k + 1]), anc_qubit)
    qc.cx(sys_qubits[closing_bit], anc_qubit)


def extract_block(op_matrix, B_target, n_total, n):
    """Extract the block-encoded matrix from the full unitary operator."""
    dim = 2 ** n_total
    best_error, best_block = float('inf'), None
    
    for anc_bit in range(n_total):
        mask = 1 << anc_bit
        idx = [i for i in range(dim) if not (i & mask)]
        block = op_matrix[np.ix_(idx, idx)]
        
        perm_list = list(permutations(range(n))) if n <= 4 else [tuple(range(n))]
        if n > 4:
            rng = np.random.default_rng(42)
            for _ in range(min(200, math.factorial(n))):
                perm_list.append(tuple(rng.permutation(n).tolist()))
        
        for p in perm_list:
            b = block[np.ix_(list(p), list(p))]
            err = np.max(np.abs(b - B_target))
            if err < best_error:
                best_error, best_block = err, b.copy()
    
    return best_block, best_error

In [19]:
def build_svd_block_encoding(A, alpha_be=None, verbose=True):
    """
    Build an SVD-decomposed block encoding circuit for matrix A.
    Returns (qc_pre, qc_best, info_dict).
    """
    n = A.shape[0]
    assert n > 0 and (n & (n - 1)) == 0, f"n={n} must be a power of 2"
    
    n_sys = int(np.log2(n))
    n_total = n_sys + 1
    
    U_A, sigmas, Vt_A = svd(A)
    spectral_norm = sigmas[0]
    
    if alpha_be is None:
        alpha_be = spectral_norm
    assert alpha_be >= spectral_norm - 1e-12, \
        f"alpha_be={alpha_be} < ||A||_2={spectral_norm:.6f}"
    
    s = sigmas / alpha_be
    B_target = A / alpha_be
    thetas = 2 * np.arccos(np.clip(s, -1, 1))
    a_ucry = build_ucry_angles(thetas, n_sys)
    
    if verbose:
        print(f"  Matrix size:     {n}x{n}")
        print(f"  System qubits:   {n_sys}")
        print(f"  Total qubits:    {n_total}")
        print(f"  ||A||_2:         {spectral_norm:.6f}")
        print(f"  alpha (BE):      {alpha_be:.6f}")
        print(f"  sigma/alpha:     {np.round(s, 6).tolist()}")
        print(f"  UCRy thetas:     {np.round(thetas, 6).tolist()}")
        print(f"  Decomposed a_k:  {np.round(a_ucry, 6).tolist()}")
    
    # Build circuit
    sys_qubits = list(range(n_sys))
    anc_qubit = n_sys
    
    qc = QuantumCircuit(n_total, name=f'BE_SVD_{n}x{n}')
    qc.append(UnitaryGate(Vt_A, label='Vdag'), sys_qubits)
    build_ucry_circuit(qc, a_ucry, sys_qubits, anc_qubit, n_sys)
    qc.append(UnitaryGate(U_A, label='U_A'), sys_qubits)
    
    if verbose:
        print(f"\n  Pre-transpilation circuit (depth {qc.depth()}):")
        print(qc.draw(output='text'))
    
    # Transpile
    pm = generate_preset_pass_manager(
        optimization_level=3, basis_gates=['cx', 'u', 'id'])
    best = None
    for _ in range(max(50, 20 * n_sys)):
        qc_t = pm.run(qc)
        if best is None or qc_t.depth() < best.depth():
            best = qc_t
    
    # Verify
    best_block, best_error = extract_block(
        Operator(best).data, B_target, n_total, n)
    
    info = {
        'U_A': U_A, 'Vt_A': Vt_A, 'sigmas': sigmas,
        'alpha_be': alpha_be, 'spectral_norm': spectral_norm,
        'B_target': B_target, 'B_extracted': best_block,
        'error': best_error, 'n_sys': n_sys, 'n_total': n_total,
        'thetas': thetas, 'a_ucry': a_ucry,
    }
    
    if verbose:
        print(f"\n  Transpiled circuit:")
        print(f"    Depth: {best.depth()},  CX: {best.count_ops().get('cx', 0)},  "
              f"U: {best.count_ops().get('u', 0)}")
        print(f"    Max element error: {best_error:.2e}")
        print(f"    Exact: {best_error < 1e-6}")
    
    return qc, best, info


def build_direct_dilation(A, alpha_be, verbose=True):
    """Direct dilation block encoding (baseline for comparison)."""
    n = A.shape[0]
    n_total = int(np.log2(n)) + 1
    
    def safe_sqrtm(M):
        w, v = np.linalg.eigh(M)
        return v @ np.diag(np.sqrt(np.maximum(w, 0))) @ v.T
    
    B = A / (alpha_be * 1.0000001)
    In = np.eye(n)
    U_BE = np.block([[B, safe_sqrtm(In - B @ B.T)],
                     [safe_sqrtm(In - B.T @ B), -B.T]])
    
    qc = QuantumCircuit(n_total)
    qc.append(UnitaryGate(U_BE), list(range(n_total)))
    
    pm = generate_preset_pass_manager(
        optimization_level=3, basis_gates=['cx', 'u', 'id'])
    best = None
    for _ in range(30):
        qc_t = pm.run(qc)
        if best is None or qc_t.depth() < best.depth():
            best = qc_t
    
    if verbose:
        print(f"  Direct dilation: depth={best.depth()}, "
              f"CX={best.count_ops().get('cx', 0)}")
    return best

In [20]:
def run_chebyshev_block_encoding(n, alpha, compare=True):
    """Full pipeline: build A, pad, block-encode, verify, compare."""
    A, m, m_pad = build_matrix(n, alpha)
    n_sys = int(np.log2(m_pad))
    
    print("=" * 65)
    print(f"  CHEBYSHEV BLOCK ENCODING: n={n}, alpha={alpha}")
    print("=" * 65)
    print(f"  D_cheb size: {m}x{m}  ->  padded to {m_pad}x{m_pad}")
    print(f"  Qubits: {n_sys} system + 1 ancilla = {n_sys + 1} total")
    
    print(f"\n--- SVD Block Encoding ---")
    qc_pre, qc_svd, info = build_svd_block_encoding(A)
    
    A_rec = info['alpha_be'] * info['B_extracted'].real
    print(f"\n  Verification: max|A_rec - A| = {np.max(np.abs(A_rec - A)):.2e}")
    
    # Show recovered matrix
    print(f"\n  Extracted block (= A / alpha_BE):")
    for row in info['B_extracted'].real:
        print(f"    [{' '.join(f'{x:10.6f}' for x in row)}]")
    
    print(f"\n  Recovered A = alpha_BE * block:")
    for row in A_rec:
        print(f"    [{' '.join(f'{x:10.4f}' for x in row)}]")
    
    if compare and m_pad <= 8:
        print(f"\n--- Direct Dilation (baseline) ---")
        qc_dir = build_direct_dilation(A, info['alpha_be'])
        
        d_s = qc_svd.depth(); cx_s = qc_svd.count_ops().get('cx', 0)
        d_d = qc_dir.depth(); cx_d = qc_dir.count_ops().get('cx', 0)
        print(f"\n  {'Method':<25} {'Depth':>6} {'CX':>6}")
        print(f"  {'---'*12}")
        print(f"  {'SVD decomposed':<25} {d_s:>6} {cx_s:>6}")
        print(f"  {'Direct dilation':<25} {d_d:>6} {cx_d:>6}")
        if d_d > 0:
            print(f"  Depth reduction: {d_d - d_s} ({(d_d-d_s)/d_d*100:.0f}%)")
    
    print(f"\n  ** NORMALISATION FACTOR: alpha_BE = {info['alpha_be']:.6f} **")
    print(f"     A = alpha_BE * <0_anc|U_BE|0_anc>")
    
    return qc_svd, info

## 5. Example: $n = 3$, $\alpha = 5$ (the original 4x4 case)

This is the matrix from the original notebook:
$A = D_{\text{cheb}}(3) - 5I = \begin{pmatrix} -5 & 1 & 0 & 3 \\ 0 & -5 & 4 & 0 \\ 0 & 0 & -5 & 6 \\ 0 & 0 & 0 & -5 \end{pmatrix}$

In [21]:
qc_3_5, info_3_5 = run_chebyshev_block_encoding(n=3, alpha=5.0)

  CHEBYSHEV BLOCK ENCODING: n=3, alpha=5.0
  D_cheb size: 4x4  ->  padded to 4x4
  Qubits: 2 system + 1 ancilla = 3 total

--- SVD Block Encoding ---
  Matrix size:     4x4
  System qubits:   2
  Total qubits:    3
  ||A||_2:         9.701731
  alpha (BE):      9.701731
  sigma/alpha:     [1.0, 0.64329, 0.511166, 0.214543]
  UCRy thetas:     [0.0, 1.744016, 2.068512, 2.709145]
  Decomposed a_k:  [1.630418, -0.596162, -0.275846, -0.75841]

  Pre-transpilation circuit (depth 9):
       ┌───────┐                                                  »
q_0: ──┤0      ├─────■─────────────────────────────────────────■──»
       │  Vdag │     │                                         │  »
q_1: ──┤1      ├─────┼────────────────────■────────────────────┼──»
     ┌─┴───────┴──┐┌─┴─┐┌──────────────┐┌─┴─┐┌──────────────┐┌─┴─┐»
q_2: ┤ Ry(1.6304) ├┤ X ├┤ Ry(-0.59616) ├┤ X ├┤ Ry(-0.27585) ├┤ X ├»
     └────────────┘└───┘└──────────────┘└───┘└──────────────┘└───┘»
«                          ┌──────┐
«q_0: 

## 6. Why the Depth is 17

The transpiled circuit for the 4x4 case has **depth 17** with **8 CX gates**.

### Pre-transpilation: 3 logical layers (depth 9)

| Layer | Operation | Gates |
|-------|-----------|-------|
| 1 | $V^\dagger$ (2-qubit unitary on $q_0, q_1$) | 1 UnitaryGate |
| 2 | UCRy (4 $R_y$ + 4 CX on $q_0/q_1 \to q_2$) | 8 gates, depth 7 |
| 3 | $U_A$ (2-qubit unitary on $q_0, q_1$) | 1 UnitaryGate |

### After transpilation to {CX, U}

| Layer | CX | U | Depth |
|-------|-----|---|-------|
| $V^\dagger$ | 2 | 4 | ~5 |
| UCRy | 4 | 4 | ~7 |
| $U_A$ | 2 | 4 | ~5 |
| **Boundary merging** | -- | absorbed | **saves ~4** |
| **Total** | **8** | **~16** | **17** |

### Why boundary merging saves depth

- The **last U gate** of $V^\dagger$ acts on $q_0/q_1$, while the **first Ry** of UCRy
  acts on $q_2$ and these execute **in parallel** (no added depth).
- Similarly, the **last CX** of UCRy targets $q_2$, while $U_A$ starts on $q_0/q_1$.
- When $\alpha_{BE} = \|A\|_2$: $\sigma_0 = \alpha_{BE}$, so
  $\theta_0 = 0$ and the first UCRy rotation is trivial and is absorbed by the transpiler.

In [22]:
# Display the transpiled circuit
print("Transpiled circuit (n=3, alpha=5):")
print(qc_3_5.draw(output='text', fold=120))

Transpiled circuit (n=3, alpha=5):
global phase: 1.5981
          ┌──────────────────┐               ┌────────────────┐                 ┌──────────────┐           »
q_0: ─────┤ U(1.4573,-π/2,0) ├───────■───────┤ U(2.638,0,π/2) ├───────■─────────┤ U(3.134,0,0) ├────────■──»
     ┌────┴──────────────────┴────┐┌─┴─┐┌────┴────────────────┴────┐┌─┴─┐┌──────┴──────────────┴─────┐  │  »
q_1: ┤ U(1.2572,-0.23187,-2.4882) ├┤ X ├┤ U(1.8034,2.3754,-2.1068) ├┤ X ├┤ U(1.1242,-1.1305,0.74192) ├──┼──»
     └─────┬───────────────┬──────┘└───┘└──────────────────────────┘└───┘└───────────────────────────┘┌─┴─┐»
q_2: ──────┤ U(1.6304,0,0) ├──────────────────────────────────────────────────────────────────────────┤ X ├»
           └───────────────┘                                                                          └───┘»
«                                                     ┌───────────────────┐                                        »
«q_0: ─────────────────────────────────────────────■──┤ U(0.1379

## 7. More Examples

In [23]:
# n=1: 2x2 matrix (1 system qubit + 1 ancilla)
qc_1, info_1 = run_chebyshev_block_encoding(n=1, alpha=2.0)

  CHEBYSHEV BLOCK ENCODING: n=1, alpha=2.0
  D_cheb size: 2x2  ->  padded to 2x2
  Qubits: 1 system + 1 ancilla = 2 total

--- SVD Block Encoding ---
  Matrix size:     2x2
  System qubits:   1
  Total qubits:    2
  ||A||_2:         2.561553
  alpha (BE):      2.561553
  sigma/alpha:     [1.0, 0.609612]
  UCRy thetas:     [0.0, 1.830451]
  Decomposed a_k:  [0.915226, -0.915226]

  Pre-transpilation circuit (depth 5):
         ┌──────┐                             ┌─────┐
q_0: ────┤ Vdag ├─────■────────────────────■──┤ U_A ├
     ┌───┴──────┴──┐┌─┴─┐┌──────────────┐┌─┴─┐└─────┘
q_1: ┤ Ry(0.91523) ├┤ X ├┤ Ry(-0.91523) ├┤ X ├───────
     └─────────────┘└───┘└──────────────┘└───┘       

  Transpiled circuit:
    Depth: 5,  CX: 2,  U: 4
    Max element error: 9.42e-17
    Exact: True

  Verification: max|A_rec - A| = 1.11e-16

  Extracted block (= A / alpha_BE):
    [ -0.780776   0.390388]
    [  0.000000  -0.780776]

  Recovered A = alpha_BE * block:
    [   -2.0000     1.0000]
    [    0

In [24]:
# n=3, alpha=0: pure differentiation (no shift)
qc_3_0, info_3_0 = run_chebyshev_block_encoding(n=3, alpha=0.0)

  CHEBYSHEV BLOCK ENCODING: n=3, alpha=0.0
  D_cheb size: 4x4  ->  padded to 4x4
  Qubits: 2 system + 1 ancilla = 3 total

--- SVD Block Encoding ---
  Matrix size:     4x4
  System qubits:   2
  Total qubits:    3
  ||A||_2:         6.723363
  alpha (BE):      6.723363
  sigma/alpha:     [1.0, 0.59494, 0.132733, 0.0]
  UCRy thetas:     [0.0, 1.86721, 2.875341, 3.141593]
  Decomposed a_k:  [1.971036, -0.533365, -0.40024, -1.037431]

  Pre-transpilation circuit (depth 9):
       ┌───────┐                                                 »
q_0: ──┤0      ├────■─────────────────────────────────────────■──»
       │  Vdag │    │                                         │  »
q_1: ──┤1      ├────┼────────────────────■────────────────────┼──»
     ┌─┴───────┴─┐┌─┴─┐┌──────────────┐┌─┴─┐┌──────────────┐┌─┴─┐»
q_2: ┤ Ry(1.971) ├┤ X ├┤ Ry(-0.53337) ├┤ X ├┤ Ry(-0.40024) ├┤ X ├»
     └───────────┘└───┘└──────────────┘└───┘└──────────────┘└───┘»
«                         ┌──────┐
«q_0: ──────────────

In [25]:
# n=7: 8x8 matrix (3 system qubits + 1 ancilla = 4 qubits)
qc_7, info_7 = run_chebyshev_block_encoding(n=7, alpha=3.0)

  CHEBYSHEV BLOCK ENCODING: n=7, alpha=3.0
  D_cheb size: 8x8  ->  padded to 8x8
  Qubits: 3 system + 1 ancilla = 4 total

--- SVD Block Encoding ---
  Matrix size:     8x8
  System qubits:   3
  Total qubits:    4
  ||A||_2:         29.375048
  alpha (BE):      29.375048
  sigma/alpha:     [1.0, 0.779807, 0.285691, 0.231424, 0.174715, 0.136563, 0.103854, 9.3e-05]
  UCRy thetas:     [0.0, 1.352879, 2.562138, 2.674509, 2.79036, 2.867611, 2.93351, 3.141407]
  Decomposed a_k:  [2.290302, -0.2188, -0.138733, -0.537589, -0.433353, -0.171394, -0.147513, -0.64292]

  Pre-transpilation circuit (depth 17):
       ┌───────┐                                                 »
q_0: ──┤0      ├─────■────────────────────────────────────────■──»
       │       │     │                                        │  »
q_1: ──┤1 Vdag ├─────┼───────────────────■────────────────────┼──»
       │       │     │                   │                    │  »
q_2: ──┤2      ├─────┼───────────────────┼──────────────────

## 8. Why the SVD Approach Gives Lower Depth

### Direct dilation: opaque black box

The QSD treats $U_{BE}$ as an arbitrary $2^k \times 2^k$ unitary, requiring $O(4^k)$ CX gates.
For $k = 3$ (4x4 matrix): typically 14-20 CX, depth 35-40.

### SVD approach: structured factorisation

Each factor operates on a **smaller subsystem**:

- $V^\dagger$ and $U_A$ are $n_{\text{sys}}$-qubit unitaries (not $(n_{\text{sys}}{+}1)$-qubit),
  so decomposition is $O(4^{n_{\text{sys}}})$ instead of $O(4^{n_{\text{sys}}+1})$ -- a **factor of 4**.
- UCRy has a **fixed structure** ($N$ CX + $N$ $R_y$) that is already optimal.
- Layer boundaries allow **parallel execution** on disjoint qubit sets.

## 9. Normalisation Factor

The block encoding stores $A/\alpha_{BE}$, not $A$ itself.
To recover the original matrix:

$$\boxed{A = \alpha_{BE} \cdot \langle 0_a | \, U_{BE} \, | 0_a \rangle}$$

where $\alpha_{BE} = \|A\|_2 = \sigma_{\max}(A)$.

### Physical meaning

- In QSVT / quantum linear systems: determines the **success probability** of ancilla measurement.
- Choosing $\alpha_{BE} > \|A\|_2$ is valid but reduces success probability.
- **Minimum** valid choice is $\alpha_{BE} = \|A\|_2$ (maximises success probability).

In [26]:
# Summary of normalisation factors
print("=" * 65)
print("  NORMALISATION FACTORS")
print("=" * 65)
print(f"  {'Case':<30} {'||A||_2':>10} {'alpha_BE':>10}")
print(f"  {'---'*18}")

for label, inf in [("n=1, alpha=2", info_1),
                   ("n=3, alpha=0 (pure D)", info_3_0),
                   ("n=3, alpha=5 (original)", info_3_5),
                   ("n=7, alpha=3", info_7)]:
    print(f"  {label:<30} {inf['spectral_norm']:>10.4f} {inf['alpha_be']:>10.4f}")

  NORMALISATION FACTORS
  Case                              ||A||_2   alpha_BE
  ------------------------------------------------------
  n=1, alpha=2                       2.5616     2.5616
  n=3, alpha=0 (pure D)              6.7234     6.7234
  n=3, alpha=5 (original)            9.7017     9.7017
  n=7, alpha=3                      29.3750    29.3750
